In [ ]:
"""Rotated MNIST known pose VAE robustness check.

This script is designed for a single Google Colab/A100 run. It creates paired
views of thresholded MNIST digits by synthetically rotating each upright digit
with two known angles. It then trains the same four pose pathways used in the
paper:

1. coordinate-conditioned decoder
2. learned-warp control
3. exact SO(2)/restricted-STN warp
4. GA rotor warp.

Outputs are written to <outdir> and zipped as rotated_mnist_results_upload.zip.
The zip contains:
    tables/table_rotated_mnist.tex
    figures/fig_rotated_mnist.pdf
    figures/fig_rotated_mnist_qualitative.pdf
    results_csv/rotated_mnist_per_seed.csv
    results_csv/rotated_mnist_config.json
"""
from __future__ import annotations

import argparse
import json
import math
import random
import shutil
import zipfile
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, List, Mapping, Sequence, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import Tensor, nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

try:
    from torchvision.datasets import MNIST
except Exception as exc:  # pragma: no cover
    raise RuntimeError("torchvision is required. In Colab, run: pip install torchvision") from exc


# -----------------------------------------------------------------------------
# Configuration and reproducibility
# -----------------------------------------------------------------------------


@dataclass
class Config:
    outdir: str = "/content/rotated_mnist_outputs"
    data_root: str = "/content/data"
    image_size: int = 32
    threshold: float = 0.15
    num_train: int = 12000
    num_val: int = 2000
    batch_size: int = 256
    epochs: int = 32
    z_dim: int = 32
    base_channels: int = 16
    learning_rate: float = 2e-3
    seeds: Tuple[int, ...] = tuple(range(10))
    device: str = "cuda"
    num_workers: int = 2


@dataclass
class LossWeights:
    beta_z: float = 1e-4
    lambda_can_bce: float = 5.0
    lambda_can_dice: float = 2.0
    lambda_view_dice: float = 1.0
    lambda_inv: float = 5.0
    lambda_conf: float = 0.02


MODEL_ORDER = ["coord", "learned_warp", "exact_so2", "ga_rotor"]
MODEL_LABELS = {
    "coord": "Coord. decoder",
    "learned_warp": "Learned warp",
    "exact_so2": "Exact $\\mathrm{SO}(2)$/STN",
    "ga_rotor": "GA rotor",
}
METRICS = [
    ("canonical_bce", "Canonical BCE $\\downarrow$", "lower"),
    ("canonical_hard_dice", "Canonical Dice $\\uparrow$", "higher"),
    ("view_bce", "View BCE $\\downarrow$", "lower"),
    ("view_hard_dice", "View Dice $\\uparrow$", "higher"),
    ("composition_mse", "Composition MSE $\\downarrow$", "lower"),
]


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# -----------------------------------------------------------------------------
# Rotation utilities
# -----------------------------------------------------------------------------


def pose_code(theta: Tensor) -> Tensor:
    return torch.stack([torch.cos(theta), torch.sin(theta)], dim=-1)


def matrix_from_angle(theta: Tensor) -> Tensor:
    """Active 2D rotation matrix for angle theta in image-normalised coordinates."""
    c = torch.cos(theta)
    s = torch.sin(theta)
    B = theta.shape[0]
    R = torch.zeros(B, 2, 2, dtype=theta.dtype, device=theta.device)
    R[:, 0, 0] = c
    R[:, 0, 1] = -s
    R[:, 1, 0] = s
    R[:, 1, 1] = c
    return R


def rotor_from_angle(theta: Tensor) -> Tensor:
    half = 0.5 * theta
    return torch.stack([torch.cos(half), torch.sin(half)], dim=-1)


def matrix_from_rotor(theta: Tensor) -> Tensor:
    """Build the same active rotation matrix through planar rotor coefficients."""
    r = rotor_from_angle(theta)
    c, s = r.unbind(dim=-1)
    cos_theta = c * c - s * s
    sin_theta = 2.0 * c * s
    B = theta.shape[0]
    R = torch.zeros(B, 2, 2, dtype=theta.dtype, device=theta.device)
    R[:, 0, 0] = cos_theta
    R[:, 0, 1] = -sin_theta
    R[:, 1, 0] = sin_theta
    R[:, 1, 1] = cos_theta
    return R


def affine_for_inverse_sampling(R: Tensor) -> Tensor:
    """Return the 2x3 affine grid matrix for inverse sampling.

    If the active action is R, the sampling coordinates are R^{-1}u = R^T u.
    """
    B = R.shape[0]
    A = torch.zeros(B, 2, 3, dtype=R.dtype, device=R.device)
    A[:, :, :2] = R.transpose(1, 2)
    return A


def render_rotated_logits(logits: Tensor, theta: Tensor, mode: str = "matrix") -> Tensor:
    if mode == "matrix":
        R = matrix_from_angle(theta)
    elif mode == "ga":
        R = matrix_from_rotor(theta)
    else:
        raise ValueError(f"unknown rotation mode: {mode}")
    A = affine_for_inverse_sampling(R)
    grid = F.affine_grid(A, logits.size(), align_corners=False)
    return F.grid_sample(logits, grid, mode="bilinear", padding_mode="zeros", align_corners=False)


@torch.no_grad()
def rotate_images(images: Tensor, theta: Tensor, threshold: float | None = None) -> Tensor:
    """Rotate a batch of images with the same operator used by the exact models."""
    rotated = render_rotated_logits(images, theta, mode="matrix")
    if threshold is not None:
        rotated = (rotated > threshold).float()
    return rotated


# -----------------------------------------------------------------------------
# Dataset construction
# -----------------------------------------------------------------------------


def load_mnist_images(root: str, train: bool, image_size: int, threshold: float) -> Tensor:
    dataset = MNIST(root=root, train=train, download=True)
    images = dataset.data.float() / 255.0  # [N, 28, 28]
    images = (images > threshold).float().unsqueeze(1)  # binary occupancy
    pad_total = image_size - 28
    if pad_total < 0 or pad_total % 2 != 0:
        raise ValueError("image_size must be an even padding of the 28x28 MNIST canvas")
    pad = pad_total // 2
    images = F.pad(images, (pad, pad, pad, pad), mode="constant", value=0.0)
    return images.contiguous()


class TensorPairs(Dataset):
    def __init__(self, canonical: Tensor, x_a: Tensor, x_b: Tensor, theta_a: Tensor, theta_b: Tensor):
        self.canonical = canonical.cpu()
        self.x_a = x_a.cpu()
        self.x_b = x_b.cpu()
        self.theta_a = theta_a.cpu()
        self.theta_b = theta_b.cpu()
        self.delta = (theta_b - theta_a).cpu()

    def __len__(self) -> int:
        return int(self.canonical.shape[0])

    def __getitem__(self, idx: int) -> Dict[str, Tensor]:
        return {
            "canonical": self.canonical[idx],
            "x_a": self.x_a[idx],
            "x_b": self.x_b[idx],
            "theta_a": self.theta_a[idx],
            "theta_b": self.theta_b[idx],
            "delta": self.delta[idx],
        }


@torch.no_grad()
def build_pairs(base_images: Tensor, n_samples: int, seed: int, device: torch.device) -> TensorPairs:
    rng = np.random.default_rng(seed)
    replace = n_samples > base_images.shape[0]
    indices = rng.choice(base_images.shape[0], size=n_samples, replace=replace)
    theta_a_np = rng.uniform(-math.pi, math.pi, size=n_samples).astype("float32")
    theta_b_np = rng.uniform(-math.pi, math.pi, size=n_samples).astype("float32")

    canonical = base_images[indices].to(device)
    theta_a = torch.from_numpy(theta_a_np).to(device)
    theta_b = torch.from_numpy(theta_b_np).to(device)

    # Build rotated binary targets in chunks to keep memory use small.
    chunks_can: List[Tensor] = []
    chunks_a: List[Tensor] = []
    chunks_b: List[Tensor] = []
    chunk = 1024
    for start in range(0, n_samples, chunk):
        end = min(start + chunk, n_samples)
        c = canonical[start:end]
        ta = theta_a[start:end]
        tb = theta_b[start:end]
        chunks_can.append(c.cpu())
        chunks_a.append(rotate_images(c, ta, threshold=0.25).cpu())
        chunks_b.append(rotate_images(c, tb, threshold=0.25).cpu())
    return TensorPairs(
        canonical=torch.cat(chunks_can, dim=0),
        x_a=torch.cat(chunks_a, dim=0),
        x_b=torch.cat(chunks_b, dim=0),
        theta_a=theta_a.cpu(),
        theta_b=theta_b.cpu(),
    )


# -----------------------------------------------------------------------------
# Models
# -----------------------------------------------------------------------------


class Encoder2D(nn.Module):
    def __init__(self, z_dim: int, image_size: int, base_channels: int):
        super().__init__()
        if image_size % 16 != 0:
            raise ValueError("image_size must be divisible by 16")
        C = base_channels
        hw = image_size // 16
        self.net = nn.Sequential(
            nn.Conv2d(1, C, 4, 2, 1), nn.SiLU(),
            nn.Conv2d(C, 2 * C, 4, 2, 1), nn.SiLU(),
            nn.Conv2d(2 * C, 4 * C, 4, 2, 1), nn.SiLU(),
            nn.Conv2d(4 * C, 8 * C, 4, 2, 1), nn.SiLU(),
            nn.Flatten(),
            nn.Linear(8 * C * hw * hw, 256), nn.SiLU(),
        )
        self.mu = nn.Linear(256, z_dim)
        self.logvar = nn.Linear(256, z_dim)

    def forward(self, x: Tensor) -> Dict[str, Tensor]:
        h = self.net(x)
        return {"mu_z": self.mu(h), "logvar_z": self.logvar(h)}


class Decoder2D(nn.Module):
    def __init__(self, input_dim: int, image_size: int, base_channels: int):
        super().__init__()
        if image_size % 16 != 0:
            raise ValueError("image_size must be divisible by 16")
        C = base_channels
        hw = image_size // 16
        self.hw = hw
        self.c4 = 8 * C
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 256), nn.SiLU(),
            nn.Linear(256, self.c4 * hw * hw), nn.SiLU(),
        )
        self.deconv = nn.Sequential(
            nn.ConvTranspose2d(8 * C, 4 * C, 4, 2, 1), nn.SiLU(),
            nn.ConvTranspose2d(4 * C, 2 * C, 4, 2, 1), nn.SiLU(),
            nn.ConvTranspose2d(2 * C, C, 4, 2, 1), nn.SiLU(),
            nn.ConvTranspose2d(C, 1, 4, 2, 1),
        )

    def forward(self, z: Tensor) -> Tensor:
        h = self.fc(z).view(-1, self.c4, self.hw, self.hw)
        return self.deconv(h)


class BasePairVAE(nn.Module):
    def reparameterize(self, mu: Tensor, logvar: Tensor) -> Tensor:
        std = torch.exp(0.5 * logvar)
        return mu + torch.randn_like(std) * std

    def encode_pair(self, x_a: Tensor, x_b: Tensor) -> Tuple[Dict[str, Tensor], Dict[str, Tensor], Tensor, Tensor]:
        enc_a = self.encoder(x_a)
        enc_b = self.encoder(x_b)
        z_a = self.reparameterize(enc_a["mu_z"], enc_a["logvar_z"])
        z_b = self.reparameterize(enc_b["mu_z"], enc_b["logvar_z"])
        z_shared = 0.5 * (z_a + z_b)
        mu_shared = 0.5 * (enc_a["mu_z"] + enc_b["mu_z"])
        return enc_a, enc_b, z_shared, mu_shared


class CoordinateVAE(BasePairVAE):
    def __init__(self, z_dim: int, image_size: int, base_channels: int):
        super().__init__()
        self.encoder = Encoder2D(z_dim, image_size, base_channels)
        self.decoder = Decoder2D(z_dim + 2, image_size, base_channels)

    def forward_pair(self, batch: Mapping[str, Tensor]) -> Dict[str, Tensor]:
        enc_a, enc_b, z_shared, mu_shared = self.encode_pair(batch["x_a"], batch["x_b"])
        p_a = pose_code(batch["theta_a"])
        p_b = pose_code(batch["theta_b"])
        p_zero = torch.zeros_like(p_a)
        p_zero[:, 0] = 1.0
        return {
            "enc_a": enc_a,
            "enc_b": enc_b,
            "canonical_logits_mean": self.decoder(torch.cat([mu_shared, p_zero], dim=-1)),
            "recon_a_mean": self.decoder(torch.cat([mu_shared, p_a], dim=-1)),
            "recon_b_mean": self.decoder(torch.cat([mu_shared, p_b], dim=-1)),
            "canonical_logits": self.decoder(torch.cat([z_shared, p_zero], dim=-1)),
            "recon_a": self.decoder(torch.cat([z_shared, p_a], dim=-1)),
            "recon_b": self.decoder(torch.cat([z_shared, p_b], dim=-1)),
        }


class AnalyticVAE(BasePairVAE):
    def __init__(self, z_dim: int, image_size: int, base_channels: int, mode: str):
        super().__init__()
        self.encoder = Encoder2D(z_dim, image_size, base_channels)
        self.decoder = Decoder2D(z_dim, image_size, base_channels)
        if mode not in {"matrix", "ga"}:
            raise ValueError(mode)
        self.mode = mode

    def forward_pair(self, batch: Mapping[str, Tensor]) -> Dict[str, Tensor]:
        enc_a, enc_b, z_shared, mu_shared = self.encode_pair(batch["x_a"], batch["x_b"])
        can = self.decoder(z_shared)
        can_mean = self.decoder(mu_shared)
        return {
            "enc_a": enc_a,
            "enc_b": enc_b,
            "canonical_logits": can,
            "canonical_logits_mean": can_mean,
            "recon_a": render_rotated_logits(can, batch["theta_a"], mode=self.mode),
            "recon_b": render_rotated_logits(can, batch["theta_b"], mode=self.mode),
            "recon_a_mean": render_rotated_logits(can_mean, batch["theta_a"], mode=self.mode),
            "recon_b_mean": render_rotated_logits(can_mean, batch["theta_b"], mode=self.mode),
        }


class LearnedWarpVAE(BasePairVAE):
    def __init__(self, z_dim: int, image_size: int, base_channels: int):
        super().__init__()
        self.encoder = Encoder2D(z_dim, image_size, base_channels)
        self.decoder = Decoder2D(z_dim, image_size, base_channels)
        self.pose_net = nn.Sequential(nn.Linear(2, 64), nn.SiLU(), nn.Linear(64, 64), nn.SiLU(), nn.Linear(64, 1))

    def predicted_theta(self, theta: Tensor) -> Tensor:
        return self.pose_net(pose_code(theta)).squeeze(-1)

    def forward_pair(self, batch: Mapping[str, Tensor]) -> Dict[str, Tensor]:
        enc_a, enc_b, z_shared, mu_shared = self.encode_pair(batch["x_a"], batch["x_b"])
        can = self.decoder(z_shared)
        can_mean = self.decoder(mu_shared)
        theta_a_hat = self.predicted_theta(batch["theta_a"])
        theta_b_hat = self.predicted_theta(batch["theta_b"])
        return {
            "enc_a": enc_a,
            "enc_b": enc_b,
            "canonical_logits": can,
            "canonical_logits_mean": can_mean,
            "recon_a": render_rotated_logits(can, theta_a_hat, mode="matrix"),
            "recon_b": render_rotated_logits(can, theta_b_hat, mode="matrix"),
            "recon_a_mean": render_rotated_logits(can_mean, theta_a_hat, mode="matrix"),
            "recon_b_mean": render_rotated_logits(can_mean, theta_b_hat, mode="matrix"),
        }


def make_model(model_name: str, cfg: Config) -> nn.Module:
    if model_name == "coord":
        return CoordinateVAE(cfg.z_dim, cfg.image_size, cfg.base_channels)
    if model_name == "learned_warp":
        return LearnedWarpVAE(cfg.z_dim, cfg.image_size, cfg.base_channels)
    if model_name == "exact_so2":
        return AnalyticVAE(cfg.z_dim, cfg.image_size, cfg.base_channels, mode="matrix")
    if model_name == "ga_rotor":
        return AnalyticVAE(cfg.z_dim, cfg.image_size, cfg.base_channels, mode="ga")
    raise ValueError(model_name)


# -----------------------------------------------------------------------------
# Losses and metrics
# -----------------------------------------------------------------------------


def kl_standard_normal(mu: Tensor, logvar: Tensor) -> Tensor:
    return 0.5 * torch.sum(torch.exp(logvar) + mu * mu - 1.0 - logvar, dim=-1)


def soft_dice(logits: Tensor, target: Tensor, eps: float = 1e-6) -> Tensor:
    probs = torch.sigmoid(logits)
    inter = (probs * target).flatten(1).sum(1)
    denom = probs.flatten(1).sum(1) + target.flatten(1).sum(1)
    return (2.0 * inter + eps) / (denom + eps)


def hard_dice(logits: Tensor, target: Tensor, eps: float = 1e-6) -> Tensor:
    probs = (torch.sigmoid(logits) > 0.5).float()
    inter = (probs * target).flatten(1).sum(1)
    denom = probs.flatten(1).sum(1) + target.flatten(1).sum(1)
    return (2.0 * inter + eps) / (denom + eps)


def confidence_penalty(logits: Tensor) -> Tensor:
    p = torch.sigmoid(logits)
    return (p * (1.0 - p)).mean()


def compute_losses(batch: Mapping[str, Tensor], out: Mapping[str, Tensor], weights: LossWeights) -> Dict[str, Tensor]:
    can = batch["canonical"]
    xa = batch["x_a"]
    xb = batch["x_b"]
    view_bce = F.binary_cross_entropy_with_logits(out["recon_a"], xa) + F.binary_cross_entropy_with_logits(out["recon_b"], xb)
    view_dice_loss = (1.0 - soft_dice(out["recon_a"], xa)).mean() + (1.0 - soft_dice(out["recon_b"], xb)).mean()
    can_bce = F.binary_cross_entropy_with_logits(out["canonical_logits_mean"], can)
    can_dice_loss = (1.0 - soft_dice(out["canonical_logits_mean"], can)).mean()
    inv = F.mse_loss(out["enc_a"]["mu_z"], out["enc_b"]["mu_z"])
    kl = kl_standard_normal(out["enc_a"]["mu_z"], out["enc_a"]["logvar_z"]).mean() + kl_standard_normal(out["enc_b"]["mu_z"], out["enc_b"]["logvar_z"]).mean()
    conf = confidence_penalty(out["canonical_logits_mean"]) + confidence_penalty(out["recon_a_mean"]) + confidence_penalty(out["recon_b_mean"])
    loss = (
        view_bce
        + weights.lambda_view_dice * view_dice_loss
        + weights.lambda_can_bce * can_bce
        + weights.lambda_can_dice * can_dice_loss
        + weights.lambda_inv * inv
        + weights.beta_z * kl
        + weights.lambda_conf * conf
    )
    return {
        "loss": loss,
        "view_bce": view_bce.detach(),
        "canonical_bce": can_bce.detach(),
        "view_hard_dice": 0.5 * (hard_dice(out["recon_a_mean"], xa).mean() + hard_dice(out["recon_b_mean"], xb).mean()).detach(),
        "canonical_hard_dice": hard_dice(out["canonical_logits_mean"], can).mean().detach(),
        "invariance": inv.detach(),
    }


@torch.no_grad()
def composition_mse(model: nn.Module, loader: DataLoader, device: torch.device) -> float:
    model.eval()
    vals: List[float] = []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model.forward_pair(batch)
        a_to_b = render_rotated_logits(out["recon_a_mean"], batch["delta"], mode="matrix")
        err = F.mse_loss(torch.sigmoid(a_to_b), torch.sigmoid(out["recon_b_mean"]))
        vals.append(float(err.cpu()))
    return float(np.mean(vals))


def run_epoch(model: nn.Module, loader: DataLoader, device: torch.device, optimizer: torch.optim.Optimizer | None, weights: LossWeights) -> Dict[str, float]:
    model.train(optimizer is not None)
    totals: Dict[str, float] = {}
    count = 0
    for batch in loader:
        batch = {k: v.to(device, non_blocking=True) for k, v in batch.items()}
        out = model.forward_pair(batch)
        losses = compute_losses(batch, out, weights)
        if optimizer is not None:
            optimizer.zero_grad(set_to_none=True)
            losses["loss"].backward()
            optimizer.step()
        bs = int(batch["x_a"].shape[0])
        count += bs
        for k, v in losses.items():
            if k == "loss":
                value = float(v.detach().cpu())
            else:
                value = float(v.cpu())
            totals[k] = totals.get(k, 0.0) + value * bs
    return {k: v / count for k, v in totals.items()}


# -----------------------------------------------------------------------------
# Reporting
# -----------------------------------------------------------------------------


def summarize(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for model in MODEL_ORDER:
        sub = df[df["model"] == model]
        row = {"model": model, "label": MODEL_LABELS[model]}
        for key, _, _ in METRICS:
            row[f"{key}_mean"] = float(sub[key].mean())
            row[f"{key}_std"] = float(sub[key].std(ddof=1))
        rows.append(row)
    return pd.DataFrame(rows)


def fmt_cell(mean: float, std: float, is_best: bool) -> str:
    cell = f"{mean:.4f} $\\pm$ {std:.4f}"
    return f"\\best{{{cell}}}" if is_best else cell


def write_latex_table(summary: pd.DataFrame, outpath: Path) -> None:
    # Bold all values tied after rounding to 4 decimals. This intentionally bolds
    # both exact SO(2)/STN and GA rotor when their displayed values match.
    best_flags: Dict[Tuple[str, str], bool] = {}
    for key, _, direction in METRICS:
        rounded = summary[f"{key}_mean"].round(4)
        best_val = rounded.min() if direction == "lower" else rounded.max()
        for _, row in summary.iterrows():
            best_flags[(row["model"], key)] = bool(round(row[f"{key}_mean"], 4) == best_val)

    lines = []
    lines.append("\\begin{tabular}{lccccc}")
    lines.append("\\toprule")
    header = "Model & " + " & ".join(label for _, label, _ in METRICS) + " \\\\"
    lines.append(header)
    lines.append("\\midrule")
    for _, row in summary.iterrows():
        cells = []
        for key, _, _ in METRICS:
            cells.append(fmt_cell(row[f"{key}_mean"], row[f"{key}_std"], best_flags[(row["model"], key)]))
        lines.append(f"{row['label']} & " + " & ".join(cells) + r" \\")
    lines.append("\\bottomrule")
    lines.append("\\end{tabular}")
    outpath.parent.mkdir(parents=True, exist_ok=True)
    outpath.write_text("\n".join(lines) + "\n")


def save_summary_figure(summary: pd.DataFrame, outpath: Path) -> None:
    labels = [MODEL_LABELS[m] for m in MODEL_ORDER]
    x = np.arange(len(labels))

    fig, axes = plt.subplots(1, 2, figsize=(9.5, 3.3))
    axes[0].bar(x, summary["view_hard_dice_mean"], yerr=summary["view_hard_dice_std"], capsize=4)
    axes[0].set_xticks(x, labels, rotation=20, ha="right")
    axes[0].set_ylabel("View Dice")
    axes[0].grid(axis="y", alpha=0.3)

    axes[1].bar(x, summary["composition_mse_mean"], yerr=summary["composition_mse_std"], capsize=4)
    axes[1].set_xticks(x, labels, rotation=20, ha="right")
    axes[1].set_ylabel("Composition MSE")
    axes[1].grid(axis="y", alpha=0.3)

    fig.tight_layout()
    outpath.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(outpath)
    plt.close(fig)


@torch.no_grad()
def save_qualitative(model: nn.Module, loader: DataLoader, device: torch.device, outpath: Path) -> None:
    model.eval()
    batch = next(iter(loader))
    batch = {k: v.to(device) for k, v in batch.items()}
    out = model.forward_pair(batch)
    idxs = [0, 1, 2, 3]
    rows = [
        ("true canon", batch["canonical"]),
        ("pred canon", (torch.sigmoid(out["canonical_logits_mean"]) > 0.5).float()),
        ("true A", batch["x_a"]),
        ("pred A", (torch.sigmoid(out["recon_a_mean"]) > 0.5).float()),
        ("true B", batch["x_b"]),
        ("pred B", (torch.sigmoid(out["recon_b_mean"]) > 0.5).float()),
    ]
    fig, axes = plt.subplots(len(idxs), len(rows), figsize=(9.0, 6.0))
    for r, idx in enumerate(idxs):
        for c, (title, tensor) in enumerate(rows):
            axes[r, c].imshow(tensor[idx, 0].detach().cpu(), cmap="gray", vmin=0, vmax=1)
            axes[r, c].axis("off")
            if r == 0:
                axes[r, c].set_title(title, fontsize=9)
    fig.tight_layout()
    outpath.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(outpath)
    plt.close(fig)


# -----------------------------------------------------------------------------
# Main experiment
# -----------------------------------------------------------------------------


def train_one(model_name: str, seed: int, cfg: Config, train_loader: DataLoader, val_loader: DataLoader, device: torch.device) -> Tuple[Dict[str, float], nn.Module]:
    set_seed(seed)
    model = make_model(model_name, cfg).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate)
    weights = LossWeights()
    for epoch in range(1, cfg.epochs + 1):
        tr = run_epoch(model, train_loader, device, opt, weights)
        va = run_epoch(model, val_loader, device, None, weights)
        print(f"seed={seed} model={model_name:12s} epoch={epoch:02d} val_view_dice={va['view_hard_dice']:.4f} val_comp=pending")
    final = run_epoch(model, val_loader, device, None, weights)
    final["composition_mse"] = composition_mse(model, val_loader, device)
    return final, model


def parse_args() -> Config:
    parser = argparse.ArgumentParser()
    parser.add_argument("--outdir", type=str, default=Config.outdir)
    parser.add_argument("--data-root", type=str, default=Config.data_root)
    parser.add_argument("--num-train", type=int, default=Config.num_train)
    parser.add_argument("--num-val", type=int, default=Config.num_val)
    parser.add_argument("--epochs", type=int, default=Config.epochs)
    parser.add_argument("--batch-size", type=int, default=Config.batch_size)
    parser.add_argument("--z-dim", type=int, default=Config.z_dim)
    parser.add_argument("--base-channels", type=int, default=Config.base_channels)
    parser.add_argument("--learning-rate", type=float, default=Config.learning_rate)
    parser.add_argument("--seeds", type=int, nargs="+", default=list(Config.seeds))
    parser.add_argument("--device", type=str, default=Config.device)
    parser.add_argument("--num-workers", type=int, default=Config.num_workers)
    args, _ = parser.parse_known_args()
    return Config(
        outdir=args.outdir,
        data_root=args.data_root,
        num_train=args.num_train,
        num_val=args.num_val,
        epochs=args.epochs,
        batch_size=args.batch_size,
        z_dim=args.z_dim,
        base_channels=args.base_channels,
        learning_rate=args.learning_rate,
        seeds=tuple(args.seeds),
        device=args.device,
        num_workers=args.num_workers,
    )


def main() -> None:
    cfg = parse_args()
    outdir = Path(cfg.outdir)
    if outdir.exists():
        shutil.rmtree(outdir)
    (outdir / "tables").mkdir(parents=True, exist_ok=True)
    (outdir / "figures").mkdir(parents=True, exist_ok=True)
    (outdir / "results_csv").mkdir(parents=True, exist_ok=True)

    device = torch.device("cuda" if cfg.device == "cuda" and torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    train_base = load_mnist_images(cfg.data_root, train=True, image_size=cfg.image_size, threshold=cfg.threshold)
    val_base = load_mnist_images(cfg.data_root, train=False, image_size=cfg.image_size, threshold=cfg.threshold)

    rows: List[Dict[str, float | int | str]] = []
    representative_model: nn.Module | None = None
    representative_loader: DataLoader | None = None

    for seed in cfg.seeds:
        print(f"\nBuilding paired Rotated MNIST data for seed {seed}")
        train_ds = build_pairs(train_base, cfg.num_train, seed=10_000 + seed, device=device)
        val_ds = build_pairs(val_base, cfg.num_val, seed=20_000 + seed, device=device)
        train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=cfg.num_workers, pin_memory=(device.type == "cuda"))
        val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=cfg.num_workers, pin_memory=(device.type == "cuda"))

        for model_name in MODEL_ORDER:
            metrics, model = train_one(model_name, seed, cfg, train_loader, val_loader, device)
            row: Dict[str, float | int | str] = {"seed": seed, "model": model_name, "label": MODEL_LABELS[model_name]}
            for key, value in metrics.items():
                if key != "loss":
                    row[key] = float(value)
            rows.append(row)
            if seed == cfg.seeds[0] and model_name == "ga_rotor":
                representative_model = model
                representative_loader = val_loader

    df = pd.DataFrame(rows)
    df.to_csv(outdir / "results_csv/rotated_mnist_per_seed.csv", index=False)
    (outdir / "results_csv/rotated_mnist_config.json").write_text(json.dumps({**asdict(cfg), "seeds": list(cfg.seeds), "loss_weights": asdict(LossWeights())}, indent=2))

    summary = summarize(df)
    summary.to_csv(outdir / "results_csv/rotated_mnist_summary.csv", index=False)
    write_latex_table(summary, outdir / "tables/table_rotated_mnist.tex")
    save_summary_figure(summary, outdir / "figures/fig_rotated_mnist.pdf")
    if representative_model is not None and representative_loader is not None:
        save_qualitative(representative_model, representative_loader, device, outdir / "figures/fig_rotated_mnist_qualitative.pdf")

    zip_path = outdir.parent / "rotated_mnist_results_upload.zip"
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for path in outdir.rglob("*"):
            if path.is_file():
                zf.write(path, arcname=str(path.relative_to(outdir)))
    print(f"\nWrote {zip_path}")
    print("All done.")


if __name__ == "__main__":
    main()


Using device: cuda


100%|██████████| 9.91M/9.91M [00:01<00:00, 5.53MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 130kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.24MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 11.9MB/s]



Building paired Rotated MNIST data for seed 0
seed=0 model=coord        epoch=01 val_view_dice=0.5588 val_comp=pending
seed=0 model=coord        epoch=02 val_view_dice=0.5818 val_comp=pending
seed=0 model=coord        epoch=03 val_view_dice=0.6083 val_comp=pending
seed=0 model=coord        epoch=04 val_view_dice=0.6129 val_comp=pending
seed=0 model=coord        epoch=05 val_view_dice=0.6267 val_comp=pending
seed=0 model=coord        epoch=06 val_view_dice=0.5935 val_comp=pending
seed=0 model=coord        epoch=07 val_view_dice=0.5801 val_comp=pending
seed=0 model=coord        epoch=08 val_view_dice=0.6226 val_comp=pending
seed=0 model=coord        epoch=09 val_view_dice=0.6309 val_comp=pending
seed=0 model=coord        epoch=10 val_view_dice=0.6549 val_comp=pending
seed=0 model=coord        epoch=11 val_view_dice=0.6585 val_comp=pending
seed=0 model=coord        epoch=12 val_view_dice=0.6594 val_comp=pending
seed=0 model=coord        epoch=13 val_view_dice=0.6512 val_comp=pending
seed